# Smart Home Energy Predictor 2026
**Google Colab Training Notebook**

Pipeline: Data Load → Feature Engineering → Train (XGBoost / RF / Ridge) → Log MLflow

> MLflow logs จะถูกบันทึกลง **Google Drive** โดยตรง (`sqlite:///MyDrive/SmartEnergy2026/runs/mlruns.db`) — ไม่ต้อง ngrok

## 1. ติดตั้ง Dependencies

In [ ]:
# ติดตั้ง dependencies ทั้งหมด
!pip install mlflow xgboost scikit-learn pandas numpy matplotlib optuna pyyaml -q
print('✅ Dependencies installed')

## 2. Mount Google Drive และ Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
import os

# --- กำหนด path หลัก ---
BASE_PATH = '/content/drive/MyDrive/SmartEnergy2026'  # เปลี่ยนตาม Drive ของคุณ
REPO_DIR  = '/content/SmartEnergy2026'

# Clone repo (ถ้ายังไม่มี)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaiwamairu/SmartEnergy2026.git {REPO_DIR}
else:
    print('Repo already exists — pulling latest...')
    !cd {REPO_DIR} && git pull

# เพิ่ม repo เข้า Python path
import sys
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'✅ Working directory: {os.getcwd()}')

## 3. ตั้งค่า MLflow Tracking URI

In [ ]:
import os

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MLflow tracking ผ่าน SQLite บน Google Drive
# ไม่ต้องใช้ ngrok หรือ server ภายนอก
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

os.makedirs(f'{BASE_PATH}/runs', exist_ok=True)
MLFLOW_TRACKING_URI = f'sqlite:///{BASE_PATH}/runs/mlruns.db'

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f'✅ MLflow Tracking URI: {MLFLOW_TRACKING_URI}')

# ทดสอบการเชื่อมต่อ
try:
    client = mlflow.tracking.MlflowClient()
    exps = client.search_experiments()
    print(f'✅ MLflow พร้อมใช้งาน — พบ {len(exps)} experiments')
except Exception as e:
    print(f'❌ MLflow error: {e}')

## 4. ดาวน์โหลด Dataset (UCI Household Power Consumption)

In [ ]:
import os
import zipfile

DATA_DIR = f'{BASE_PATH}/data'
os.makedirs(DATA_DIR, exist_ok=True)

RAW_FILE = f'{DATA_DIR}/household_power_consumption.txt'
PROCESSED_FILE = f'{DATA_DIR}/processed_hourly.csv'

if not os.path.exists(RAW_FILE):
    print('กำลังดาวน์โหลด UCI dataset...')
    !wget -q -O /tmp/household.zip https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip
    with zipfile.ZipFile('/tmp/household.zip', 'r') as z:
        z.extractall(DATA_DIR)
    print('✅ Dataset downloaded')
else:
    print(f'✅ Dataset already exists: {RAW_FILE}')

## 5. Preprocess Data (Resample → Hourly)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.data.data_loader import load_raw_data, resample_hourly

if not os.path.exists(PROCESSED_FILE):
    print('กำลัง preprocess ข้อมูล...')
    df_raw = load_raw_data(RAW_FILE)
    print(f'Raw data shape: {df_raw.shape} | Date range: {df_raw.index[0]} → {df_raw.index[-1]}')

    df_hourly = resample_hourly(df_raw)
    df_hourly.to_csv(PROCESSED_FILE)
    print(f'✅ Processed data saved: {PROCESSED_FILE}')
    print(f'Hourly shape: {df_hourly.shape}')
else:
    import pandas as pd
    df_hourly = pd.read_csv(PROCESSED_FILE, index_col=0, parse_dates=True)
    print(f'✅ Loaded existing processed data: {df_hourly.shape}')

print('\nSample data:')
df_hourly[['Global_active_power']].tail()

## 6. ปรับ Config Path ให้ชี้ไป Google Drive

In [ ]:
import yaml

CONFIG_DIR = f'{REPO_DIR}/configs'
BASE_CFG_PATH = f'{CONFIG_DIR}/base.yaml'

# โหลดและอัปเดต base_path ใน config ให้ตรงกับ Drive
with open(BASE_CFG_PATH, 'r') as f:
    base_cfg = yaml.safe_load(f)

base_cfg['data']['base_path'] = BASE_PATH
base_cfg['mlflow']['tracking_uri'] = MLFLOW_TRACKING_URI

# เขียน config ชั่วคราวสำหรับ run นี้
RUNTIME_CFG_PATH = '/tmp/base_runtime.yaml'
with open(RUNTIME_CFG_PATH, 'w') as f:
    yaml.dump(base_cfg, f)

print('✅ Config updated with Drive paths')
print(f"  base_path: {base_cfg['data']['base_path']}")
print(f"  tracking_uri: {base_cfg['mlflow']['tracking_uri']}")

## 7. เทรน Baseline — Ridge Regression

In [ ]:
from src.pipelines.train_pipeline import run_training

print('='*60)
print('Training: Ridge Regression (Baseline)')
print('='*60)

result_ridge = run_training(
    model_name='ridge',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ Ridge done | Run ID: {result_ridge['run_id']}")

## 8. เทรน Random Forest

In [ ]:
print('='*60)
print('Training: Random Forest')
print('='*60)

result_rf = run_training(
    model_name='random_forest',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ Random Forest done | Run ID: {result_rf['run_id']}")

## 9. เทรน XGBoost (Main Model)

In [ ]:
print('='*60)
print('Training: XGBoost Regressor')
print('='*60)

result_xgb = run_training(
    model_name='xgboost',
    data_path=PROCESSED_FILE,
    tracking_uri=MLFLOW_TRACKING_URI,
    version=1,
    config_dir=CONFIG_DIR,
)

print(f"\n✅ XGBoost done | Run ID: {result_xgb['run_id']}")

## 10. เปรียบเทียบผลลัพธ์ทุกโมเดล

In [ ]:
import pandas as pd

results = {
    'Ridge (Baseline)': result_ridge,
    'Random Forest': result_rf,
    'XGBoost': result_xgb,
}

rows = []
for name, r in results.items():
    m = r['test_metrics']
    rows.append({
        'Model': name,
        'RMSE': round(m['rmse'], 4),
        'MAE': round(m['mae'], 4),
        'R²': round(m['r2'], 4),
        'Pass Threshold': '✅' if r['passed_threshold'] else '❌',
        'Run ID': r['run_id'][:8] + '...',
    })

df_results = pd.DataFrame(rows).sort_values('RMSE')
print('\n' + '='*70)
print('MODEL COMPARISON — Test Set Results')
print('='*70)
print(df_results.to_string(index=False))
print(f'\nProduction threshold: RMSE < 0.15')

## 11. Register Best Model ใน MLflow Registry

In [ ]:
# หาโมเดลที่ดีที่สุดและผ่าน threshold
best = min(results.items(), key=lambda x: x[1]['test_metrics']['rmse'])
best_name, best_result = best

if best_result['passed_threshold']:
    MODEL_REGISTRY_NAME = 'SmartEnergyPredictor'
    model_uri = f"runs:/{best_result['run_id']}/model"

    registered = mlflow.register_model(model_uri, MODEL_REGISTRY_NAME)
    print(f'✅ Model registered: {MODEL_REGISTRY_NAME} v{registered.version}')
    print(f'   Best model: {best_name}')
    print(f'   Test RMSE: {best_result["test_metrics"]["rmse"]:.4f}')
    print(f'   Run ID: {best_result["run_id"]}')
else:
    print(f'⚠️  Best model ({best_name}) RMSE = {best_result["test_metrics"]["rmse"]:.4f} ≥ threshold 0.15')
    print('   ยังไม่ผ่านเกณฑ์ — ลอง Hyperparameter Tuning ใน Cell ถัดไป')

## 12. (Optional) Hyperparameter Tuning ด้วย Optuna

In [ ]:
import optuna
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from src.utils.config_loader import load_config
from src.utils.seed_utils import set_global_seed
from src.data.data_loader import load_processed_data
from src.data.preprocessor import prepare_features, time_series_split, fit_scaler, apply_scaler

# โหลดข้อมูลและ features (ใช้ซ้ำจาก cell ก่อน)
cfg = load_config('xgboost', config_dir=CONFIG_DIR)
set_global_seed(cfg['project']['seed'])

df = load_processed_data(PROCESSED_FILE)
X, y = prepare_features(df, cfg)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(
    X, y, cfg['data']['test_size'], cfg['data']['val_size']
)
X_train_s, scaler = fit_scaler(X_train)
X_val_s = apply_scaler(X_val, scaler)

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'objective': 'reg:squarederror',
        'n_jobs': -1,
        'verbosity': 0,
    }
    model = XGBRegressor(**params, random_state=cfg['project']['seed'])
    model.fit(X_train_s, y_train)
    val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val_s)))
    return val_rmse

# บันทึก study ลง Drive เพื่อให้รันต่อได้ถ้า Colab ตัด
STUDY_DB = f'sqlite:///{BASE_PATH}/runs/optuna_xgb.db'
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
    storage=STUDY_DB,
    study_name='xgb_energy_tuning',
    load_if_exists=True,
)

N_TRIALS = 50  # เพิ่มได้ถึง 100
print(f'Starting Optuna search — {N_TRIALS} trials...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best RMSE: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

# บันทึก best params
import json
best_params_path = f'{BASE_PATH}/runs/best_xgb_params.json'
with open(best_params_path, 'w') as f:
    json.dump({'best_value': study.best_value, 'best_params': study.best_params}, f, indent=2)
print(f'Saved to: {best_params_path}')

## 13. เทรน XGBoost ด้วย Best Params จาก Optuna

In [ ]:
import json, yaml
from src.utils.mlflow_helpers import setup_mlflow, make_run_name, hash_file, log_pip_freeze, log_scaler_params
from src.data.preprocessor import apply_scaler
from src.pipelines.train_pipeline import compute_metrics, plot_actual_vs_pred, plot_feature_importance
import tempfile, mlflow, mlflow.xgboost

# โหลด best params
with open(f'{BASE_PATH}/runs/best_xgb_params.json') as f:
    best_data = json.load(f)

best_params = best_data['best_params']
best_params.update({'objective': 'reg:squarederror', 'n_jobs': -1, 'verbosity': 1})

setup_mlflow(MLFLOW_TRACKING_URI, cfg['mlflow']['experiment_name'])
mlflow.xgboost.autolog(log_models=True, log_datasets=False)

X_test_s = apply_scaler(X_test, scaler)
run_name = make_run_name('xgb_tuned', 2)

with mlflow.start_run(run_name=run_name):
    mlflow.set_tags({'algo': 'xgboost_tuned', 'optuna_trials': N_TRIALS})
    log_scaler_params(scaler)

    model_tuned = XGBRegressor(**best_params, random_state=42)
    model_tuned.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
                    early_stopping_rounds=30, verbose=50)

    test_metrics = compute_metrics(y_test, model_tuned.predict(X_test_s))
    mlflow.log_metrics({f'test_{k}': v for k, v in test_metrics.items()})

    print(f"\nTuned XGBoost — Test RMSE: {test_metrics['rmse']:.4f} | MAE: {test_metrics['mae']:.4f} | R²: {test_metrics['r2']:.4f}")

    with tempfile.TemporaryDirectory() as tmpdir:
        plot_actual_vs_pred(y_test, model_tuned.predict(X_test_s), run_name,
                            f'{tmpdir}/actual_vs_pred_plot.png')
        mlflow.log_artifact(f'{tmpdir}/actual_vs_pred_plot.png')
        plot_feature_importance(model_tuned, list(X.columns), f'{tmpdir}/feature_importance.png')
        mlflow.log_artifact(f'{tmpdir}/feature_importance.png')
        log_pip_freeze(tmpdir)

    tuned_run_id = mlflow.active_run().info.run_id

threshold = cfg['baseline']['rmse_threshold']
if test_metrics['rmse'] < threshold:
    print(f'✅ RMSE {test_metrics["rmse"]:.4f} < {threshold} → Register ได้')
    registered = mlflow.register_model(f'runs:/{tuned_run_id}/model', 'SmartEnergyPredictor')
    print(f'   Registered as version {registered.version}')
else:
    print(f'⚠️  RMSE {test_metrics["rmse"]:.4f} ≥ {threshold}')